In [2]:
import torch
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# 1. 데이터 불러오기
df = sns.load_dataset("diamonds").dropna()  # 결측치 제거

In [9]:
# 2. X, y 나누기
X = df.drop(columns="price")
y = df["price"]

In [10]:
# 3. 범주형 변수 인코딩
X = pd.get_dummies(X)  # 범주형 변수 원-핫 인코딩
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)  # species -> 숫자

In [11]:
# 4. 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [14]:
# 5. train/test 나누기
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=None)

In [15]:
# 6. 텐서로 변환
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)  # 다중 분류이므로 long
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [16]:
# 7. 데이터로더 생성
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [17]:
# 8. 모델 정의 (nn.Module 상속)
import torch.nn as nn

class PenguinClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(PenguinClassifier, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim)
        )

    def forward(self, x):
        return self.model(x)


In [18]:
# 9. 손실함수 및 옵티마이저
import torch.optim as optim

input_dim = X_train_tensor.shape[1]
output_dim = len(label_encoder.classes_)

model = PenguinClassifier(input_dim, output_dim)
criterion = nn.CrossEntropyLoss()  # 다중 클래스 분류
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [19]:
# 10. 학습 함수
def train_model(epochs=50):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

In [20]:
# 11. 평가 함수
def evaluate_model():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"Accuracy on test set: {accuracy:.2f}%")

In [21]:
# 12. 실행

if __name__ == "__main__":
    train_model(epochs=50)
    evaluate_model()

Epoch 1/50, Loss: 8.4553
Epoch 2/50, Loss: 7.5998
Epoch 3/50, Loss: 7.1242
Epoch 4/50, Loss: 6.7898
Epoch 5/50, Loss: 6.5266
Epoch 6/50, Loss: 6.3039
Epoch 7/50, Loss: 6.1084
Epoch 8/50, Loss: 5.9395
Epoch 9/50, Loss: 5.7918
Epoch 10/50, Loss: 5.6625
Epoch 11/50, Loss: 5.5471
Epoch 12/50, Loss: 5.4432
Epoch 13/50, Loss: 5.3507
Epoch 14/50, Loss: 5.2640
Epoch 15/50, Loss: 5.1881
Epoch 16/50, Loss: 5.1197
Epoch 17/50, Loss: 5.0558
Epoch 18/50, Loss: 4.9950
Epoch 19/50, Loss: 4.9413
Epoch 20/50, Loss: 4.8882
Epoch 21/50, Loss: 4.8305
Epoch 22/50, Loss: 4.7745
Epoch 23/50, Loss: 4.7290
Epoch 24/50, Loss: 4.6826
Epoch 25/50, Loss: 4.6413
Epoch 26/50, Loss: 4.6031
Epoch 27/50, Loss: 4.5633
Epoch 28/50, Loss: 4.5305
Epoch 29/50, Loss: 4.4985
Epoch 30/50, Loss: 4.4693
Epoch 31/50, Loss: 4.4383
Epoch 32/50, Loss: 4.4108
Epoch 33/50, Loss: 4.3890
Epoch 34/50, Loss: 4.3629
Epoch 35/50, Loss: 4.3383
Epoch 36/50, Loss: 4.3183
Epoch 37/50, Loss: 4.2983
Epoch 38/50, Loss: 4.2789
Epoch 39/50, Loss: 4.

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# 1. 메시스러운 패턴이 있는 가짜 데이터 생성
def generate_messi_data(samples=1000):
    X = np.random.rand(samples, 10).astype(np.float32)  # 10개의 메시 특징
    y = []

    for i in range(samples):
        # 메시성 점수: 드리블(0), 시야(1), 탈압박(5)에 비중
        score = X[i][0]*0.5 + X[i][1]*0.3 + X[i][5]*0.2
        if score > 0.8:
            y.append(2)  # 메시기모띠
        elif score > 0.5:
            y.append(1)  # 메시
        else:
            y.append(0)  # 일반인

    return torch.tensor(X), torch.tensor(y, dtype=torch.long)

# 2. 모델 정의 - 메시급 GOAT 아키텍처
class LionelMessiNet(nn.Module):
    def __init__(self):
        super(LionelMessiNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(10, 64),  # 드리블 능력 입력
            nn.ReLU(),
            nn.Linear(64, 32),  # 킬패
            nn.ReLU(),
            nn.Linear(32, 3)    # 3개의 클래스: 일반인, 메시, 메시기모띠
        )

    def forward(self, x):
        return self.model(x)

# 3. 학습 함수
def train(model, loader, criterion, optimizer, epochs=50):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for inputs, labels in loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

# 4. 평가 함수
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"🧠 Test Accuracy: {acc:.2f}%")

# 5. 실행
if __name__ == "__main__":
    # 데이터 생성 및 분할
    X, y = generate_messi_data(1000)
    dataset = TensorDataset(X, y)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # 모델, 손실 함수, 최적화 설정
    model = LionelMessiNet()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("🎓 메시기모띠 모델 학습 시작")
    train(model, train_loader, criterion, optimizer, epochs=50)

    print("\n🧪 평가 중...")
    evaluate(model, test_loader)


🎓 메시기모띠 모델 학습 시작
Epoch [1/50] Loss: 1.0174
Epoch [2/50] Loss: 0.8911
Epoch [3/50] Loss: 0.8191
Epoch [4/50] Loss: 0.7577
Epoch [5/50] Loss: 0.6647
Epoch [6/50] Loss: 0.5612
Epoch [7/50] Loss: 0.4732
Epoch [8/50] Loss: 0.4212
Epoch [9/50] Loss: 0.3831
Epoch [10/50] Loss: 0.3486
Epoch [11/50] Loss: 0.3244
Epoch [12/50] Loss: 0.3047
Epoch [13/50] Loss: 0.2814
Epoch [14/50] Loss: 0.2633
Epoch [15/50] Loss: 0.2496
Epoch [16/50] Loss: 0.2315
Epoch [17/50] Loss: 0.2166
Epoch [18/50] Loss: 0.2028
Epoch [19/50] Loss: 0.1944
Epoch [20/50] Loss: 0.1855
Epoch [21/50] Loss: 0.1727
Epoch [22/50] Loss: 0.1667
Epoch [23/50] Loss: 0.1572
Epoch [24/50] Loss: 0.1531
Epoch [25/50] Loss: 0.1472
Epoch [26/50] Loss: 0.1393
Epoch [27/50] Loss: 0.1371
Epoch [28/50] Loss: 0.1296
Epoch [29/50] Loss: 0.1225
Epoch [30/50] Loss: 0.1234
Epoch [31/50] Loss: 0.1178
Epoch [32/50] Loss: 0.1148
Epoch [33/50] Loss: 0.1113
Epoch [34/50] Loss: 0.1142
Epoch [35/50] Loss: 0.1058
Epoch [36/50] Loss: 0.1017
Epoch [37/50] Loss: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class CristianoRonaldoNet(nn.Module):
    def __init__(self):
        super(CristianoRonaldoNet, self).__init__()
        self.power = nn.Linear(10, 128)    # 힘과 속도
        self.relu = nn.ReLU()
        self.accuracy = nn.Linear(128, 64) # 슈팅 정확도
        self.speed = nn.Linear(64, 32)     # 스피드
        self.finish = nn.Linear(32, 1)     # 골 결정력

    def forward(self, x):
        x = self.power(x)
        x = self.relu(x)
        x = self.accuracy(x)
        x = self.relu(x)
        x = self.speed(x)
        x = self.relu(x)
        x = self.finish(x)
        return x

# 모델 만들기
model = CristianoRonaldoNet()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("앙 호날두띠 모델 준비 완료! ⚡️")

앙 호날두띠 모델 준비 완료! ⚡️
